# Mid term project
## My goal

I am interested in analysis of the field related to the aviation. My current job is related to this industry that is why it may be of special intrest to me. Selectoion of the dataset is more due to the curiosity then for any practical reason. One practical side effect of this work will be cleaning data which I will be albe to use outside of this project  I am interested is there any corelation to the size and GDP of country to its air passanger traffic. Also I am interested to see how covid affected those relations. In the process I am coursious if I will find some interesting data related to the passenger traffic and airports themselves. As an additional activity I will also analysie the physical properites of the runwaays on each airport. I expect to find some corelation in regard to civilian and military airports


## Acquiring initial data

In [288]:
import requests
import pandas as pd
import shutil
import os
import re
import inspect

##### Functions

In [289]:
def add_func_name():
    if True:
        return '\n\t\t("'+inspect.currentframe().f_back.f_code.co_name+'" function.)'

In [290]:
def print_red(text):
    red_text = f"\033[91m{text}\033[0m"
    print(red_text)

In [291]:
def download_file(filename,request):
    if re.search(r"Backup_data", filename)==None and (request.status_code == 200):
        with open(filename, 'wb') as f:
            f.write(request.content)
            print(f'"{filename}" downloaded.'+add_func_name())


In [292]:
def try_to_download_file(filename, response):
    try:
        response.raise_for_status()
        data_folder = "Data"
        file_path= "/".join([data_folder, filename])
    
        print(f'Request was successful. \n"{filename}" can be downloaded.'+ add_func_name() )

    except requests.HTTPError as e:
        file_path = "/".join(["Backup_data", filename])
        print(print_red('An error occurred:' + str(e)))
        print(print_red(f'Use backup file: {file_path}'))

    return file_path

In [293]:
def save_and_return_dataframe_from_csv(filename,request):
    download_file(filename,request)
    df = pd.read_csv(filename)
    print(f'"{filename}" saved and read into the DataFrame.' + add_func_name())  
    return df

In [303]:
def extract_zip(zip_file):
    data_folder = "__".join(["Extracted", zip_file[:-4]])
    data_folder_path = "/".join(["Data",data_folder])
    
    zip_file_path = "/".join(["Data",zip_file])
    # Create folder for extracted files if do not exists
    os.makedirs(data_folder_path, exist_ok=True) 
    
    # Unzip ZIP file
    if os.path.exists(zip_file_path):
        shutil.unpack_archive(zip_file_path, data_folder_path,'zip')     
        print(f'"{zip_file}" extracted into \\{data_folder_path} ' + add_func_name())
    else:
        zip_file_path = "/".join(["Backup_data",zip_file])
        shutil.unpack_archive(zip_file_path, data_folder_path,'zip')     
        
        print(print_red(f"USING BACKUP FILE: {zip_file_path}"))
    
    
    return data_folder_path

In [304]:
def return_data_filename__worldbank(search_string, data_folder):
    files = shutil.os.listdir(data_folder)
    for f_name in files:
        if re.search(search_string, f_name) and not re.match(r"Metadata", f_name):
            print("CSV data filename selected." + add_func_name())
            return f_name
    print("ERROR: No match." + add_func_name())
    return None

In [305]:
def get_worldbank_zip_into_csv_data_file_path(key_string, zip_filename):
    "key_string - is the string which is specific for the worldbank CSV data extracted for specific indicator"
    
    # Extracting zip file and passing extract folder
    extract_folder = extract_zip(zip_filename)
    
    # Finding csv data file and returning CSV data file name
    data_filename = return_data_filename__worldbank(key_string, extract_folder)
    file_path='/'.join([extract_folder,data_filename])
    
    # Returning relative path to the CSV data file
    print(f'The CSV data path is: "{file_path}"' + add_func_name())
    return file_path

## Downloading data

### Airports data

In [297]:
# Creating data folder
os.makedirs("Data", exist_ok=True) 

In [298]:
## Getting airport database
URL1 = "https://ourairports.com/airports.csv"
response1 = requests.get(URL1)

In [299]:
filename = "airports_databas_raw.csv"
file_path = try_to_download_file(filename, response1)
airports= save_and_return_dataframe_from_csv(file_path, response1)
airports.shape

Request was successful. 
"airports_databas_raw.csv" can be downloaded.
		("try_to_download_file" function.)
"Data/airports_databas_raw.csv" downloaded.
		("download_file" function.)
"Data/airports_databas_raw.csv" saved and read into the DataFrame.
		("save_and_return_dataframe_from_csv" function.)


(35161, 23)

### Runways data

In [300]:
# Getting runway data
URL2 = 'https://davidmegginson.github.io/ourairports-data/runways.csv'
response2 = requests.get(URL2)

filename ='runways_raw.csv'
file_path = try_to_download_file(filename, response2)
runways = save_and_return_dataframe_from_csv(file_path, response2)
runways.shape

Request was successful. 
"runways_raw.csv" can be downloaded.
		("try_to_download_file" function.)
"Data/runways_raw.csv" downloaded.
		("download_file" function.)
"Data/runways_raw.csv" saved and read into the DataFrame.
		("save_and_return_dataframe_from_csv" function.)


(44914, 20)

### WORLD BANK - Passengers traffic per country (PSGR)

In [301]:
params={
    'downloadformat':'csv'
    }
URL3='https://api.worldbank.org/v2/en/indicator/IS.AIR.PSGR'
response3 = requests.get(URL3,params=params)
response3.url

'https://api.worldbank.org/v2/en/indicator/IS.AIR.PSGR?downloadformat=csv'

In [306]:
filename = "worldbank_PSGR.zip"
file_path = try_to_download_file(filename, response3)

download_file(file_path, response3)

file_path = get_worldbank_zip_into_csv_data_file_path("PSGR", filename)

worldbank_PSGR = pd.read_csv(file_path, skiprows=3)
worldbank_PSGR.shape

Request was successful. 
"worldbank_PSGR.zip" can be downloaded.
		("try_to_download_file" function.)
"Data/worldbank_PSGR.zip" downloaded.
		("download_file" function.)
"worldbank_PSGR.zip" extracted into \Data/Extracted__worldbank_PSGR 
		("extract_zip" function.)
CSV data filename selected.
		("return_data_filename__worldbank" function.)
The CSV data path is: "Data/Extracted__worldbank_PSGR/API_IS.AIR.PSGR_DS2_en_csv_v2_5607143.csv"
		("get_worldbank_zip_into_csv_data_file_path" function.)


(266, 68)

### WORLD BANK - GDP per capita (GDP_PCAP)
https://data.worldbank.org/indicator/NY.GDP.PCAP.CD

In [ ]:
# API access
params = {
    "downloadformat":'csv'
}
URL4 = "https://api.worldbank.org/v2/en/indicator/NY.GDP.PCAP.KD"
response4 = requests.get(URL4, params=params)
response4.url

In [ ]:
filename = "worldbank_GDP_PCAP.zip"
file_path = try_to_download_file(filename, response4)

download_file(file_path, response4)

file_path = get_worldbank_zip_into_csv_data_file_path("PCAP", filename)

worldbank__GDP_PCAP = pd.read_csv(file_path, skiprows=3)
worldbank__GDP_PCAP.shape

### WORLD BANK - population

In [ ]:
params = {
    "downloadformat":'csv'
}
URL5 = "https://api.worldbank.org/v2/en/indicator/SP.POP.TOTL"
response5 = requests.get(URL5, params=params)
response5.url

In [ ]:
filename = "worldbank_POP_TOTL.zip"
file_path = try_to_download_file(filename, response5)

download_file(file_path, response5)

file_path = get_worldbank_zip_into_csv_data_file_path("TOTL", filename)

worldbank__GDP_PCAP = pd.read_csv(file_path, skiprows=3)
worldbank__GDP_PCAP.shape

# Initial analysis of the data

At this point there are following dataframes uploaded into memory:
    airports - contains world airport data
    runways - contains more precise data related to the runways at each airport, such as length and width of the runway
    worldbank_PSGR - Passengers traffic per country data
    worldbank__GDP_PCAP - contains Gross Domestic product per capita for most of the countries
    

In [ ]:
# Checking the Ai
[item for item in airports.head(0)]

In [ ]:
# Exploring how the data looks like
airports.head(2)

In [ ]:
[item for item in runways.head(0)]

In [ ]:
# Testing access to selected airport by its 4-letter ICAO code. In this case EPWA means Warsaw Chopin Airport , Poland
airport_ident = 'EPWA'
df = runways
# Boolean indexing to filter rows
filtered_rows = df[df['airport_ident'] == airport_ident]
filtered_rows

In [ ]:
worldbank_PSGR

In [ ]:
## Converting from "wide format" to long format
df = worldbank_PSGR
df_melted = df.melt(
    id_vars=['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code'], 
    value_vars=[str(i) for i in range(1960, 2023)],  # list the years here
    var_name='Year', 
    value_name='Value'
)

# filter out rows where 'Value' is NaN
df_melted = df_melted.dropna(subset=['Value'])
df_melted

In [ ]:
## Printing countries names to see check what convention is used to call each 
countries = worldbank_PSGR["Country Name"]
[print(x) for x in countries]

In [ ]:
## Preparing PSGR_number to display on the year/PSGR_number initial plot

# Convert 'Year' to integer
df_melted['Year'] = df_melted['Year'].astype(int)

# List of countries to keep
countries_to_keep = ['Italy',"Germany", "Netherlands","United Kingdom" ]

# Filter dataframe
df_filtered = df_melted[df_melted['Country Name'].isin(countries_to_keep)]

In [ ]:
## Importing modules for PLOTTING
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

In [ ]:
### PLOTTING PSGR_NUM vs YEAR for SELECTED COUNTRIES
g=sns.relplot(kind="line",x="Year",y="Value", hue="Country Name",errorbar=None, data=df_filtered )

# Access the underlying matplotlib axes
ax = g.axes[0, 0]

# Format the y-axis tick labels with commas every thousand
ax.yaxis.set_major_formatter(mticker.StrMethodFormatter('{x:,.0f}'))

# Display the plot
plt.show()

### Getting GDP per capita
By looking at the general chart of passenger numbers by country and my general knowledge regarding GDP of each country it is possible that the some kind of factor POPULATIONxGDP may infulence the passenger travel in some regions

Selection of the GDP per capita type:

"Comparisons of national wealth are frequently made on the basis of nominal GDP and savings (not just income), which do not reflect differences in the cost of living in different countries (see List of countries by GDP (nominal) per capita); hence, using a PPP basis is arguably more useful when comparing generalized differences in living standards between economies because PPP takes into account the relative cost of living and the inflation rates of the countries, rather than using only exchange rates, which may distort the real differences in income." 
Wikipedia , accesed on 06.07.2023 (https://en.wikipedia.org/wiki/List_of_countries_by_GDP_(PPP)_per_capita)